# Model predictions

In [2]:
cd ../tcrsat

/Users/yang.an/PhD/Straub_An_et_al_2026_The-total-epitope-specific-T-cell-receptor-solution-space/TCRprediction/tcrsat


In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
import seaborn as sns
import os
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import confusion_matrix
import sklearn.metrics as metrics

import sys
sys.path.append('..')

from tcrsat.models.predictor import Predictor
from tcrsat.data import TCRSatDataset


/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/lightning_fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [ ]:
datas = []
all_metric_results = []
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_path = 'saved_models'
criterion = 'auc'
mod = 'full'

for epi in ['GP33', 'M45', 'siinfekl_reactive']:
    print('-'*10)
    print(epi)
    seq_data = pd.read_csv(f'../data/{epi}_with_background.csv', index_col=0)
    for i in range(5):
        split = f'{i}_group_stratified'


        ckpt_path = f'../saved_models/{epi}/{epi}_{split}.ckpt'           
        model = Predictor.load_from_checkpoint(ckpt_path, map_location=torch.device(device))

        model.eval()
        model = model.to(device)

        data_config = model.data_config.copy()
        val_dataset = TCRSatDataset(config=data_config, split=None, tokenizer=model.tokenizer, data=seq_data)
        val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=128, shuffle=False)

        outputs = defaultdict(list)
        with torch.no_grad():
            for batch in tqdm(val_dataloader):
                batch = [item.to(device) for item in batch]
                x, attention_mask, y = batch
                y_hat_logits = model(x, attention_mask)
                y_hat = model.transform_predict(y_hat_logits)

                outputs['y_hat'].append(y_hat.detach())
                outputs['y'].append(y.detach())

            y_hat = torch.cat(outputs['y_hat'])
            y = torch.cat(outputs['y'])

            y_hat = y_hat.cpu()
            y = y.cpu()
            seq_data[f'prediction_{epi}_{i}'] = y_hat

            split_mask = seq_data[data_config['split_col']].to_numpy()
            metric_results = {}
            metric_results['epitope'] = epi
            metric_results['modality'] = mod
            metric_results['split'] = split

            for split in ['train', 'val', 'test']:
                mask = (split_mask == split)
                fpr, tpr, thresholds = metrics.roc_curve(y[mask], y_hat[mask])
                optimal_threshold_auc = thresholds[np.argmax(tpr - fpr)]
                precision, recall, thresholds = metrics.precision_recall_curve(y[mask], y_hat[mask])
                optimal_threshold_prc = thresholds[np.nanargmax(2 * precision * recall / (precision + recall))]

                for metric_name, metric in model.evaluation.items():
                    metric = metric.to('cpu')
                    metric_results[f'new_{split}_{metric_name}'] = metric(y_hat[mask], y.int()[mask]).float().item()
            all_metric_results.append(metric_results)

    # Ensemble evaluation
    seq_data[f'prediction_{epi}_ensemble'] = seq_data[[f'prediction_{epi}_{i}' for i in range(5)]].mean(axis=1)
    y_hat = torch.tensor(seq_data[f'prediction_{epi}_ensemble'].values)

    split_mask = seq_data[data_config['split_col']].to_numpy()
    metric_results = {}
    metric_results['epitope'] = epi
    metric_results['modality'] = mod
    metric_results['split'] = 'ensemble'

    for split in ['test']:
        mask = (split_mask == split)
        fpr, tpr, thresholds = metrics.roc_curve(y[mask], y_hat[mask])
        optimal_threshold_auc = thresholds[np.argmax(tpr - fpr)]
        precision, recall, thresholds = metrics.precision_recall_curve(y[mask], y_hat[mask])
        optimal_threshold_prc = thresholds[np.nanargmax(2 * precision * recall / (precision + recall))]

        for metric_name, metric in model.evaluation.items():
            metric = metric.to('cpu')
            metric_results[f'new_{split}_{metric_name}'] = metric(y_hat[mask], y[mask].int()).float().item()
        all_metric_results.append(metric_results)
    os.makedirs('../results', exist_ok=True)
    seq_data.to_csv(f'../results/{epi}_prediction_scores.csv')

----------
GP33


Some weights of the model checkpoint at facebook/esm2_t12_35M_UR50D were not used when initializing EsmModel: ['lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.bias']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using LoRA
trainable params: 368640 || all params: 34361521 || trainable%: 1.072827946120313


Some weights of the model checkpoint at facebook/esm2_t30_150M_UR50D were not used when initializing EsmModel: ['lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.bias']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using LoRA
trainable params: 19660800 || all params: 168456281 || trainable%: 11.671158761957948
